### Dataset and Task Metadata

In [ ]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="home_credit_default_stability",
    dataset_year="2024",
    domain_str="finance",
    # Data Source
    dataset_source="Kaggle",
    original_dataset_source_download_link="https://www.kaggle.com/competitions/home-credit-credit-risk-model-stability",
    download_description="""
We get the data from the Kaggle competition.

kaggle competitions download -c home-credit-credit-risk-model-stability
mkdir -p local-data-warehouse/home_credit_default_stability && mv home-credit-credit-risk-model-stability.zip local-data-warehouse/home_credit_default_stability/ && cd local-data-warehouse/home_credit_default_stability/ && unzip home-credit-credit-risk-model-stability.zip && rm home-credit-credit-risk-model-stability.zip && rm -rf csv_files && rm -rf parquet_files/test && rm sample_submission.csv feature_definitions.csv
""",
    # References
    academic_reference_bibtex=r"""@misc{Herman2024HomeCreditCreditRiskModelStability,
  author = {Daniel Herman and Tomas Jelinek and Walter Reade and Maggie Demkin and Addison Howard},
  title  = {Home Credit - Credit Risk Model Stability},
  year   = {2024},
  howpublished = {\url{https://kaggle.com/competitions/home-credit-credit-risk-model-stability}},
  note   = {Kaggle competition}
}
""",
    academic_reference_bibtex_key="Herman2024HomeCreditCreditRiskModelStability",
    license="Kaggle Competition Rules",
    data_tags=["Non-IID", "Temporal"],
    curation_comments="""
We start with the data from Kaggle and follow the preprocessing from TabRed (https://github.com/yandex-research/tabred/tree/main/preprocessing#homecredit-default-stability-homecredit-20), which in turn follows two Kaggle solutions (https://www.kaggle.com/competitions/home-credit-credit-risk-model-stability/discussion/507946, https://www.kaggle.com/code/yuuniekiri/fork-of-home-credit-catboost-inference).

- We follow TabRed and use temporal splits for the task. The Kaggle experts used various strategies and most commonly StratifiedGroupKFold to align offline CV with the temporal-split data on the leaderboard. That is, the split used for training can be StratifiedGroupKFold, but not for testing.
- Note, the original competition was heavily influenced by metric hacking, which is not relevant for our offline benchmark tasks. Thus, results are not directly comparable ot transferable.
- We change the preprocessing from Kaggle and TabRed in specific steps: (A) we depart from the TabRed and Kaggle solutions in that we do not drop high-cardinality string columns, (B) we do not drop month and week, as it was only dropped due to the metric leak in the competition, (C) we only drop constant columns if they are constant w.r.t. NaN and non-NaN values, (D) we do not perform ordinal encoding and drop minor categories, (E) we do not sub-sample the data, (F) we keep the date column without transformations for the pipelines to handle, and (G) the code from TabRed is outdated and does not function the same anymore when it comes to down-casting string and categorical data of the data from Kaggle, we fix this by treating categorical data as categorical following the original Kaggle scripts, moreover, we had to remove getting the std() of a date column, which is not supported anymore.
- We drop case_id as it is already collapsed and thus does not contain extra information.
- We cast all object and string columns to be categorical. Note that they are anonymized and string preprocessing likely does not add a lot of value as a result.
- We drop name-related columns as they do not hold relevant information. These were otherwise dropped as high-cardinality string columns in TabRed or on Kaggle.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="target",
    problem_type="binary_classification",
    # There exists a custom metric see https://www.kaggle.com/competitions/home-credit-credit-risk-model-stability
    # However, we decide to not use it, as it was seen to be not stable and was able to be clearly exploited for modelling in the competition.
    objective_metric_name="roc_auc",
    stratify_on="target",
    time_on="date_decision",
)

## Preprocessing

Start by running `run_large_data_preprocessing.py` outside this notebook to get the merged data in place.

In [ ]:
import pandas as pd

df = pd.read_parquet(dataset_mold.path / "merged_input_data.parquet")
print("Loaded data shape:", df.shape)

# Drop constant columns
df = df.drop(columns=[
    # ID column (already collapsed and thus no extra information
    "case_id",
    # Name columns that are missing or have no affect due to case-id related grouping and anonymization
    "last_name_4917606M",
    "first_name_4917606M",
    "last_name_4527232M",
    "first_name_4527232M",
    "last_employername_160M",
    "first_employername_160M",
])
df["date_decision"] = pd.to_datetime(df["date_decision"], format="%Y-%m-%d")
as_cat_type = [
    "target",
    *list(df.columns[
    (df.dtypes == "object") | (df.dtypes == "string")
    ])
]
df[as_cat_type] = df[as_cat_type].astype("category")
df = df.reset_index(drop=True)

## Data Checks

In [ ]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
    duplicate_column_check=False,
)

In [ ]:
# Sample Rows
df_head

In [ ]:
# Feature Summary
summary

In [ ]:
# Numeric Feature Statistics
numeric_stats

In [ ]:
# Categorical Feature Statistics
cat_stats

In [ ]:
# Target Distribution
target_df

## Task Curation

In [ ]:
from data_foundry.schema import PredictiveMLSplitsMetadata

split_time = pd.Timestamp("2020-05-01")
# Test: all data from 2020-05-01 onwards
test_idx = df.index[
    df["date_decision"] >= split_time
].to_numpy().tolist()

# Train: all data before that
train_idx = df.index[
    df["date_decision"] < split_time
].to_numpy().tolist()


# Size and class distribution checks
print("Train size:", len(train_idx), " | Test size:", len(test_idx))
print("Train target distribution:\n", df.loc[train_idx, task_mold.target_column_name].value_counts(normalize=True))
print("Test target distribution:\n", df.loc[test_idx, task_mold.target_column_name].value_counts(normalize=True))

splits = {
    0: {
        0: (train_idx, test_idx),
    }
}

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="We follow TabRed and simulate a use case of a model being refit every 4 to 5 months. We create one test split using 4,5 months of data (2020-05-01 to 2020-10-05) for testing and all the previous data for training.",
    splits=splits,
    time_horizon=5, # round up
    time_horizon_unit="months",
)

## Export

In [ ]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)